#**Импорты**

In [1]:
# Подключение класса для создания нейронной сети прямого распространения
from tensorflow.keras.models import Sequential
# Подключение класса для создания полносвязного слоя
from tensorflow.keras.layers import Dense
# Подключение оптимизатора
from tensorflow.keras.optimizers import Adam
# Подключение утилит для to_categorical
from tensorflow.keras import utils
# Подключение библиотеки для загрузки изображений
from tensorflow.keras.preprocessing import image
# Подключение библиотеки для работы с массивами
import numpy as np
# Подключение библиотек для отрисовки изображений
import matplotlib.pyplot as plt
# Подключение модуля для работы с файлами
import os
# Для красивых таблиц
import pandas as pd
# Вывод изображения в ноутбуке, а не в консоли или файле
%matplotlib inline

# **Загрузка данных**

In [ ]:
# Загрузка датасета из облака
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip', None, quiet=True)

# Распаковываем архив hw_light.zip в папку hw_light
!unzip -q hw_light.zip

# **Загрузка изображений в массивы**

In [3]:
# Путь к директории с базой
base_dir = '/content/hw_light'
# Создание пустого списка для загрузки изображений обучающей выборки
x_train = []
# Создание списка для меток классов
y_train = []
# Задание высоты и ширины загружаемых изображений
img_height = 20
img_width = 20
# Перебор папок в директории базы
for patch in os.listdir(base_dir):
    # Перебор файлов в папках
    for img in os.listdir(base_dir + '/' + patch):
        # Добавление в список изображений текущей картинки
        x_train.append(image.img_to_array(image.load_img(base_dir + '/' + patch + '/' + img,
                                                    target_size=(img_height, img_width),
                                                    color_mode='grayscale')))
        # Добавление в массив меток, соответствующих классам
        if patch == '0':
            y_train.append(0)
        elif patch == '3':
            y_train.append(1)
        else:
            y_train.append(2)

# Преобразование в numpy-массив загруженных изображений и меток классов
x_train = np.array(x_train)
y_train = np.array(y_train)
# Вывод размерностей
print('Размер массива x_train', x_train.shape)
print('Размер массива y_train', y_train.shape)

Размер массива x_train (302, 20, 20, 1)
Размер массива y_train (302,)


# **Подготовка данных для нейросети**

In [4]:
# Нормализация пикселей (делим на 255)
x_train = x_train.astype('float32') / 255.0

# Преобразование меток в one-hot encoding (3 класса)
y_train_cat = utils.to_categorical(y_train, num_classes=3)

# Изменение формы: из (302, 20, 20, 1) в (302, 400)
x_train_flat = x_train.reshape(x_train.shape[0], -1)

print('После подготовки:')
print('x_train_flat shape:', x_train_flat.shape)
print('y_train_cat shape:', y_train_cat.shape)

После подготовки:
x_train_flat shape: (302, 400)
y_train_cat shape: (302, 3)


# **Проведение экспериментов**

In [5]:
# Параметры для перебора
neurons_list = [10, 100, 5000]
activations = ['relu', 'linear']
batch_sizes = [10, 100, 1000]

# Количество эпох (фиксируем)
epochs = 10

# Список для хранения результатов
results = []

# Вложенные циклы
for neurons in neurons_list:
    for activation in activations:
        for batch_size in batch_sizes:
            print(f"\n--- Изначальные данные: нейронов={neurons}, активация={activation}, размер батча={batch_size} ---")

            # Создаём модель
            model = Sequential()
            # Скрытый слой
            model.add(Dense(neurons, input_dim=400, activation=activation))
            # Выходной слой (softmax для классификации)
            model.add(Dense(3, activation='softmax'))

            # Компиляция
            model.compile(loss='categorical_crossentropy',
                          optimizer=Adam(),
                          metrics=['accuracy'])

            # Обучение
            history = model.fit(x_train_flat, y_train_cat,
                                batch_size=batch_size,
                                epochs=epochs,
                                verbose=0)
            model.save_weights('l1z1.weights.h5')

            # Оценка точности на обучающей выборке
            loss, acc = model.evaluate(x_train_flat, y_train_cat, verbose=1)

            # Сохраняем результат
            results.append({
                'neurons': neurons,
                'activation': activation,
                'batch_size': batch_size,
                'accuracy': acc
            })

            print(f"Точность: {acc:.4f}")


--- Изначальные данные: нейронов=10, активация=relu, размер батча=10 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.8069 - loss: 0.5671
Точность: 0.8245

--- Изначальные данные: нейронов=10, активация=relu, размер батча=100 ---
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7069 - loss: 0.7559
Точность: 0.5464

--- Изначальные данные: нейронов=10, активация=relu, размер батча=1000 ---
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.6180 - loss: 0.8978
Точность: 0.5464

--- Изначальные данные: нейронов=10, активация=linear, размер батча=10 ---
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8561 - loss: 0.5432
Точность: 0.8278

--- Изначальные данные: нейронов=10, активация=linear, размер батча=100 ---
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6964 - loss: 0.7802
Точность: 0.5762

--- Изначальные данные: нейронов=10, активация=linear, размер батча=1000 ---
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.5223 - loss: 1.0114
Точность: 0.5497

--- Изначальные данные: нейронов=100, активация=relu, размер батча=10 -

# **Сводная таблица**

In [6]:
# Создаём DataFrame для красивой таблицы
df = pd.DataFrame(results)

# Сортируем для удобства
df = df.sort_values(by=['neurons', 'activation', 'batch_size'])

# Выводим таблицу
print("Результаты экспериментов:")
print(df.to_string(index=False))

# Можно также вывести сводную таблицу для наглядности
pivot = df.pivot_table(values='accuracy',
                       index=['neurons', 'activation'],
                       columns='batch_size',
                       aggfunc='first')
print("\nСводная таблица точности (размер батча по столбцам):")
print(pivot)

Результаты экспериментов:
 neurons activation  batch_size  accuracy
      10     linear          10  0.827815
      10     linear         100  0.576159
      10     linear        1000  0.549669
      10       relu          10  0.824503
      10       relu         100  0.546358
      10       relu        1000  0.546358
     100     linear          10  0.880795
     100     linear         100  0.745033
     100     linear        1000  0.599338
     100       relu          10  0.933775
     100       relu         100  0.745033
     100       relu        1000  0.738411
    5000     linear          10  0.662252
    5000     linear         100  0.576159
    5000     linear        1000  0.403974
    5000       relu          10  0.900662
    5000       relu         100  0.807947
    5000       relu        1000  0.456954

Сводная таблица точности (размер батча по столбцам):
batch_size              10        100       1000
neurons activation                              
10      linear      0.82